In [1]:
# Uncomment to install climdata in Google Colab or other environments
# !pip install climdata

# MSWX Dataset — climdata Example

**MSWX** (Multi-Source Weather) is a global, daily, 0.1° meteorological dataset based on the ERA5 reanalysis, bias-corrected against in-situ station observations. It is distributed via **Google Drive** and accessed in climdata through a Google Service Account.

**Available variables:**

| Variable   | Description                        | Units   |
|------------|------------------------------------|---------|
| `tasmin`   | Daily minimum air temperature      | °C      |
| `tasmax`   | Daily maximum air temperature      | °C      |
| `tas`      | Daily mean air temperature         | °C      |
| `pr`       | Daily precipitation                | mm/day  |
| `rsds`     | Downward shortwave radiation       | W/m²    |
| `hurs`     | Relative humidity                  | %       |
| `sfcWind`  | Wind speed                         | m/s     |

> **Prerequisites:** A Google Service Account JSON key file with read access to the MSWX Google Drive folders is required for downloading data.

## 1. Setup & Imports

In [2]:
from climdata import ClimData
from climdata.datasets.MSWX import MSWXmirror
import xarray as xr
import pandas as pd
import logging

logging.basicConfig(
    level=logging.INFO,
    format="%(levelname)s | %(message)s",
    force=True,
)

## 2. Configuration

Set up the configuration overrides. The `google_service_account` path must point to your Google Service Account JSON key.

In [ ]:
overrides = [
    "dataset=mswx",                             # Select the MSWX dataset
    "lat=52",                                   # Target latitude
    "lon=13",                                   # Target longitude (Berlin, Germany)
    "time_range.start_date=1989-01-01",         # Start date for data extraction
    "time_range.end_date=2024-12-31",           # End date for data extraction
    "variables=[tasmin,tasmax,pr]",             # Variables to extract
    "data_dir=./data",                          # Local directory to store downloaded files
    "dsinfo.mswx.params.google_service_account=./.climdata_conf/service.json",  # Path to service account key
]

extractor = ClimData(overrides=overrides)

## 3. Download & Load Data

MSWX data is stored as daily NetCDF files on Google Drive (one file per day per variable). The `fetch()` method checks for existing local files and only downloads missing ones. The `load()` method reads all files into a single `xarray.Dataset`.

In [ ]:
mswx = MSWXmirror(extractor.cfg)

# Extract a single point (nearest grid cell)
mswx.extract(point=(extractor.cfg.lon, extractor.cfg.lat))

# Load each variable
#ds_tasmin = mswx.load(variable="tasmin")
#ds_tasmax = mswx.load(variable="tasmax")
ds_hurs     = mswx.load(variable="hurs")

print(ds_tasmin)

INFO | file_cache is only supported with oauth2client<4.0.0


📂 10327 exist, 2822 missing — fetching hurs from Drive...
⬇️ Downloading 1996266.nc ...
   → Download 100% complete
⬇️ Downloading 1996265.nc ...
   → Download 100% complete
⬇️ Downloading 1996264.nc ...
   → Download 100% complete
⬇️ Downloading 1996263.nc ...
   → Download 100% complete
⬇️ Downloading 1996262.nc ...
   → Download 100% complete
⬇️ Downloading 1996261.nc ...
   → Download 100% complete
⬇️ Downloading 1996260.nc ...
   → Download 100% complete
⬇️ Downloading 1996259.nc ...
   → Download 100% complete
⬇️ Downloading 1996258.nc ...
   → Download 100% complete
⬇️ Downloading 1996257.nc ...
   → Download 100% complete
⬇️ Downloading 1996256.nc ...
   → Download 100% complete
⬇️ Downloading 1996255.nc ...
   → Download 100% complete
⬇️ Downloading 1996254.nc ...
   → Download 100% complete
⬇️ Downloading 1996253.nc ...
   → Download 100% complete
⬇️ Downloading 1996252.nc ...
   → Download 100% complete
⬇️ Downloading 1996251.nc ...
   → Download 100% complete
⬇️ Downloading

## 4. Inspect the Dataset

In [ ]:
# Inspect one variable
ds_tasmin

In [ ]:
# Convert to pandas DataFrame for quick inspection
df_tasmin = ds_tasmin.to_dataframe().reset_index()
df_tasmin.head()

## 5. Spatial Extraction — Bounding Box

In [ ]:
overrides_box = [
    "dataset=mswx",
    "time_range.start_date=2010-01-01",
    "time_range.end_date=2010-03-31",
    "variables=[tasmax,pr]",
    "data_dir=./data",
    "dsinfo.mswx.params.google_service_account=./.climdata_conf/service.json",
]

extractor_box = ClimData(overrides=overrides_box)
mswx_box = MSWXmirror(extractor_box.cfg)

# Extract a bounding box (Germany)
mswx_box.extract(box={
    "lon_min": 5.8,
    "lon_max": 15.0,
    "lat_min": 47.3,
    "lat_max": 55.1
})

ds_box = mswx_box.load(variable="tasmax")
print(ds_box)

## 6. Run the Full Workflow via ClimData

Use `ClimData.run_workflow()` to chain extraction, optional imputation, climate index calculation, and saving to NetCDF in one call.

In [ ]:
overrides_wf = [
    "dataset=mswx",
    "lat=52",
    "lon=13",
    "time_range.start_date=2010-01-01",
    "time_range.end_date=2010-12-31",
    "variables=[tasmin,tasmax,pr]",
    "data_dir=./data",
    "dsinfo.mswx.params.google_service_account=./.climdata_conf/service.json",
    "index=tn10p",      # Cold nights climate extreme index
    "impute=BRITS",     # Deep-learning gap-filling (optional)
]

seq = ["extract", "impute", "calc_index", "to_nc"]

extractor_wf = ClimData(overrides=overrides_wf)
result = extractor_wf.run_workflow(actions=seq)
result

## 7. Save to CSV

In [ ]:
mswx.save_csv("mswx_point_2010.csv")
print("Saved to mswx_point_2010.csv")